# Dự đoán sống sót

## Khởi tạo thí nghiệm

### Khai báo thư viện

In [28]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython import display

from scipy.stats import spearmanr
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score

### Tham số thực nghiệm

In [37]:
params_cfg = {
    # "action"   : "train_feat01",  
    # "feat_path": "../../exps/featbase_251028/data.npz",
    "seed"    : 42, # Set random seed
    "exp_dir" : os.path.abspath('../exps/output'),
    'exp_name': 'result_baseline',
    "data_dir": os.path.abspath("../exps/feature1"),
    "verbose" : True,
    "k_fold": 10,
}
params_cfg.update(**{
    "save_dir": os.path.abspath(f'{params_cfg["exp_dir"]}/{params_cfg["exp_name"]}')
})

for v in params_cfg:
    print(f'+ {v}: {params_cfg[v]}')

globals().update(**params_cfg)

+ seed: 42
+ exp_dir: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output
+ exp_name: result_baseline
+ data_dir: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\feature1
+ verbose: True
+ k_fold: 10
+ save_dir: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_baseline


### Load dữ liệu đã tiền xử lí 

In [30]:
df_train = pd.read_excel(f'{params_cfg["data_dir"]}/train_df_preprocess.xlsx')
df_test = pd.read_excel(f'{params_cfg["data_dir"]}/test_df_preprocess.xlsx')

data = np.load(f'{params_cfg["data_dir"]}/feat_preprocess.npz', allow_pickle=True)

# Load feature columns
feat_cols = np.load(f'{params_cfg["data_dir"]}/feature_columns.npz', allow_pickle=True)
feature_columns = feat_cols['feature_columns']

# Chuyển về DataFrame
x= pd.DataFrame(data['x_train'], columns=feature_columns)
y = pd.Series(data['y_train'], name='Survived')
x_test_final = pd.DataFrame(data['x_test'], columns=feature_columns)

display.display(df_train.sample(5))
if params_cfg["verbose"]:
    print("-"*10, "information", "-"*10)
    print(f'train shape: {df_train.shape}')
    print(f'test shape: {df_test.shape}')
    print(f'train-col: {set(df_train.columns)}')
    print(f'test-col: {set(df_test.columns)}')
    print("Union:", set(df_train.columns).intersection(set(df_test.columns)))
    print("Difference:", set(df_train.columns).difference(set(df_test.columns)))

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,...,Embarked,Title,FamilySize,IsChild,IsMother,HasCabin,Age*Pclass,Deck,Fare_Pclass,TicketPrefix
69,70,0,3,"Kink, Mr. Vincenz",male,3.295837,2,0,315151,2.268252,...,S,Mr,3,0,0,0,9.887511,U,0.756084,NUM
809,810,1,1,"Chambers, Mrs. Norman Campbell (Bertha Griggs)",female,3.526361,1,0,113806,3.990834,...,S,Mrs,2,0,0,1,3.526361,E,3.990834,NUM
40,41,0,3,"Ahlin, Mrs. Johan (Johanna Persdotter Larsson)",female,3.713572,1,0,7546,2.348991,...,S,Mrs,2,0,0,0,11.140716,U,0.782997,NUM
259,260,1,2,"Parrish, Mrs. (Lutie Davis)",female,3.931826,0,1,230433,3.295837,...,S,Mrs,2,0,1,0,7.863651,U,1.647918,NUM
531,532,0,3,"Toufik, Mr. Nakli",male,3.295837,0,0,2641,2.107689,...,C,Mr,1,0,0,0,9.887511,U,0.702563,NUM


---------- information ----------
train shape: (891, 21)
test shape: (418, 20)
train-col: {'IsMother', 'Deck', 'Cabin', 'Fare', 'PassengerId', 'Parch', 'SibSp', 'HasCabin', 'IsChild', 'Age*Pclass', 'Age', 'Survived', 'FamilySize', 'TicketPrefix', 'Fare_Pclass', 'Ticket', 'Sex', 'Pclass', 'Name', 'Embarked', 'Title'}
test-col: {'IsMother', 'Deck', 'Cabin', 'Fare', 'PassengerId', 'Parch', 'SibSp', 'HasCabin', 'IsChild', 'Age*Pclass', 'Age', 'FamilySize', 'TicketPrefix', 'Fare_Pclass', 'Ticket', 'Sex', 'Pclass', 'Name', 'Embarked', 'Title'}
Union: {'IsMother', 'Deck', 'Cabin', 'Fare', 'PassengerId', 'Parch', 'SibSp', 'HasCabin', 'IsChild', 'Age*Pclass', 'Age', 'FamilySize', 'TicketPrefix', 'Fare_Pclass', 'Ticket', 'Sex', 'Pclass', 'Name', 'Embarked', 'Title'}
Difference: {'Survived'}


### Preprocess pipeline

In [31]:
# Preprocessor (ColumnTransformer)
num_features = [
    'Age','Fare','FamilySize','Fare_Pclass','Age*Pclass'
]
cat_features = [
    'Pclass','Sex','Embarked','Title','IsChild','IsMother',
    'Deck','HasCabin','TicketPrefix'
]

num_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer([('num', num_transformer, num_features), ('cat', cat_transformer, cat_features)])

## Models Training

**Các mô hình dùng để phân tích**: `Logistic Regression`, `Random Forest`, `XGBoost`, `SVM`

Định nghĩa các pipeline mô hình học máy (dùng chung preprocessor, khác models)
- `Logistic Regression`: mô hình tuyến tính cơ bản (baseline)
- `Random Forest`: tập hợp nhiều cây quyết định, giúp giảm overfitting
- `XGBoost`: mô hình boosting mạnh mẽ, cho hiệu suất cao nhất
- `SVM`: bộ phân loại phi tuyến, phù hợp với ranh giới phức tạp

In [32]:
# --- Khởi tạo các pipeline ---
models = {
    'Logistic Regression': Pipeline([
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(max_iter=1000,random_state=42,C=1.0,solver='lbfgs'))
    ]),

    'Random Forest': Pipeline([
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(random_state=42, n_jobs=-1,n_estimators=500, max_depth=5,min_samples_split=5, min_samples_leaf=3))
    ]),

    'XGBoost': Pipeline([
        ('preprocessor', preprocessor),
        ('model', XGBClassifier(random_state=42,eval_metric='logloss',n_jobs=-1,n_estimators=120,max_depth=2,learning_rate=0.08,subsample=0.6,colsample_bytree=0.6,reg_lambda=3,reg_alpha=2))
    ]),

    'SVM': Pipeline([
        ('preprocessor', preprocessor),
        ('model', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42))
    ])
}

- Mỗi mô hình đều kết hợp với cùng bộ xử lý dữ liệu (preprocessor) để đảm bảo đầu vào nhất quán.
- Lưu trong dictionary models giúp dễ huấn luyện và so sánh kết quả giữa các thuật toán.

### Đánh giá từng model bằng cross-validation

In [33]:
cv = StratifiedKFold(n_splits=params_cfg["k_fold"], shuffle=True, random_state=42)

results = {'Model': [], 'Metric': [], 'Score': []}

for name, model in models.items():
    acc_scores = cross_val_score(model, x, y, cv=cv, scoring='accuracy')
    f1_scores  = cross_val_score(model, x, y, cv=cv, scoring='f1')
    auc_scores = cross_val_score(model, x, y, cv=cv, scoring='roc_auc')

    # Lưu chi tiết từng lần CV
    for s in acc_scores:
        results['Model'].append(name)
        results['Metric'].append('Accuracy')
        results['Score'].append(s)
    for s in f1_scores:
        results['Model'].append(name)
        results['Metric'].append('F1 Score')
        results['Score'].append(s)
    for s in auc_scores:
        results['Model'].append(name)
        results['Metric'].append('ROC AUC')
        results['Score'].append(s)

    # Tính trung bình và độ lệch chuẩn
    acc_mean, acc_std = acc_scores.mean(), acc_scores.std()
    f1_mean, f1_std = f1_scores.mean(), f1_scores.std()
    auc_mean, auc_std = auc_scores.mean(), auc_scores.std()

    print(f"{name} CV Results:")
    print(f"  Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
    print(f"  F1 Score: {f1_mean:.4f} ± {f1_std:.4f}")
    print(f"  ROC AUC:  {auc_mean:.4f} ± {auc_std:.4f}")
    print("-" * 40)

Logistic Regression CV Results:
  Accuracy: 0.8192 ± 0.0338
  F1 Score: 0.7568 ± 0.0479
  ROC AUC:  0.8750 ± 0.0377
----------------------------------------
Random Forest CV Results:
  Accuracy: 0.8271 ± 0.0321
  F1 Score: 0.7529 ± 0.0557
  ROC AUC:  0.8710 ± 0.0459
----------------------------------------
XGBoost CV Results:
  Accuracy: 0.8159 ± 0.0365
  F1 Score: 0.7477 ± 0.0587
  ROC AUC:  nan ± nan
----------------------------------------
SVM CV Results:
  Accuracy: 0.8350 ± 0.0227
  F1 Score: 0.7652 ± 0.0406
  ROC AUC:  0.8727 ± 0.0339
----------------------------------------


- Sử dụng StratifiedKFold (10-fold CV) để đánh giá 4 mô hình khác nhau.
- Tính 3 chỉ số: Accuracy, F1 Score, ROC AUC cho từng mô hình.
- Lưu và in kết quả trung bình ± độ lệch chuẩn để so sánh hiệu suất và độ ổn định giữa các mô hình.

## Xuất ra file kết quả

In [39]:
# Tạo thư mục lưu kết quả
os.makedirs(params_cfg["save_dir"], exist_ok=True)

submissions = {}

# Train và dự đoán cho từng model
for name, model in models.items():
    print(f"Training {name}...")
    
    # Fit model trên toàn bộ training data
    model.fit(x, y)
    
    # Dự đoán trên test set
    predictions = model.predict(x_test_final)
    
    # Lưu vào dictionary với tên file phù hợp
    filename = f'submission_{name.lower().replace(" ", "_")}.csv'
    submissions[filename] = predictions
    
    print(f"✓ {name} training completed!")
    print("-" * 40)

# Lưu tất cả submissions ra file CSV
print("\n📁 Saving submission files...")
for filename, predictions in submissions.items():
    submission = pd.DataFrame({
        'PassengerId': df_test['PassengerId'],
        'Survived': predictions.astype(int)
    })

    filepath = os.path.join(params_cfg["save_dir"], filename)
    submission.to_csv(filepath, index=False)
    print(f"✓ Saved: {filepath}")

print(f"\n🎉 Created {len(submissions)} submission files in: {params_cfg['save_dir']}")

Training Logistic Regression...
✓ Logistic Regression training completed!
----------------------------------------
Training Random Forest...
✓ Random Forest training completed!
----------------------------------------
Training XGBoost...
✓ XGBoost training completed!
----------------------------------------
Training SVM...
✓ SVM training completed!
----------------------------------------

📁 Saving submission files...
✓ Saved: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_baseline\submission_logistic_regression.csv
✓ Saved: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_baseline\submission_random_forest.csv
✓ Saved: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_baseline\submission_xgboost.csv
✓ Saved: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_baseline\submission_svm.csv

🎉 Created 4 submission files in: d:\ML_git\Machine_Learning_basic\Titanic_final\exps\output\result_baseline
